# Risk of Bias: RoB2, ROBINS-I, AMSTAR-2 — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-2/risk-of-bias.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicai/learn-notebooks/blob/main/notebooks/evidence-synthesis/15-2/risk-of-bias.ipynb)

> Companion to the **Risk of Bias** article. Risk of bias is *systematic* error from how a study
> was run — a loaded coin more data can't fix. Every tool shares one pattern: a reviewer answers
> **signaling questions** -> a fixed **algorithm** maps them to a **domain judgment** ->
> domains combine into an **overall** judgment (**RoB2 uses worst-domain logic**). Here you build
> a deterministic **RoB2** evaluator and watch the same trial flip from High to Some concerns on
> a single answer.

## What you'll build

1. A deterministic **RoB2** evaluator — five domain rules plus **worst-domain** combine logic.
2. The article's worked example: an open-label physio trial that is **High** risk for self-reported **pain**, but only **Some concerns** for **mortality** — the same trial, per outcome.
3. A six-trial risk-of-bias table showing how **one High domain sinks a whole trial**.

**No API key needed.** A pure-Python rule engine (only the standard-library `enum`) — the reviewer answers the questions, the code applies the algorithm identically every time.

### Setup (nothing to install)

The one import is the standard-library `enum`, used to make "worst domain" a simple `max()`.

In [1]:
from enum import IntEnum
print("ready — pure Python, no API key, no installs")

ready — pure Python, no API key, no installs


## The judgment scale (ordered so max = worst)

RoB2 judges each domain **Low / Some concerns / High**. We back them with an ordered `IntEnum`
so the entire worst-domain combine rule collapses to `max(domains.values())`. The signaling
answers are `Y` (yes), `PY` (probably yes), `PN` (probably no), `N` (no).

In [2]:
class Judg(IntEnum):          # ordered so max() = "worst"
    LOW = 0
    SOME = 1
    HIGH = 2

LABEL = {Judg.LOW: "Low", Judg.SOME: "Some concerns", Judg.HIGH: "High"}

# signaling answers: 'Y' yes, 'PY' probably yes, 'PN' probably no, 'N' no
YESISH = {"Y", "PY"}
NOISH = {"N", "PN"}

print("judgments :", [LABEL[j] for j in Judg])
print("yes-ish   :", YESISH, "  no-ish:", NOISH)

judgments : ['Low', 'Some concerns', 'High']
yes-ish   : {'Y', 'PY'}   no-ish: {'PN', 'N'}


## The five domain rules

Each domain turns its signaling answers into a label via a **fixed** rule (a teaching
simplification of the official RoB2 algorithm; the *shape* is the real thing). The classic
**High** is D4: a subjective outcome measured by an unblinded assessor.

In [3]:
def d1_randomization(a: dict) -> Judg:
    """D1: random sequence + concealed allocation + no baseline imbalance."""
    if a["random_sequence"] in NOISH or a["allocation_concealed"] in NOISH:
        return Judg.HIGH
    if a["baseline_balanced"] in NOISH:
        return Judg.SOME
    return Judg.LOW


def d2_deviations(a: dict) -> Judg:
    """D2: awareness of assignment + whether deviations affected the outcome."""
    if a["deviations_affected_outcome"] in YESISH:
        return Judg.HIGH
    if a["participants_aware"] in YESISH:
        return Judg.SOME          # aware but no outcome-affecting deviation
    return Judg.LOW


def d3_missing_data(a: dict) -> Judg:
    """D3: was outcome data available for (nearly) all, and is missingness benign?"""
    if a["data_for_nearly_all"] in YESISH:
        return Judg.LOW
    if a["missingness_related_to_outcome"] in YESISH:
        return Judg.HIGH          # differential dropout tied to the outcome
    return Judg.SOME


def d4_measurement(a: dict) -> Judg:
    """D4: subjective outcome + unblinded assessor is the classic HIGH."""
    if a["assessor_blinded"] in YESISH:
        return Judg.LOW
    if a["outcome_subjective"] in YESISH:
        return Judg.HIGH          # unblinded assessor + subjective outcome
    return Judg.SOME              # unblinded but objective outcome


def d5_reported_result(a: dict) -> Judg:
    """D5: was this result pre-specified, or picked after seeing the data?"""
    if a["prespecified"] in YESISH:
        return Judg.LOW
    return Judg.HIGH if a["selected_from_many"] in YESISH else Judg.SOME

print("five domain rules defined (D1..D5)")

five domain rules defined (D1..D5)


## Worst-domain logic + the full assessment

The **overall** judgment is as bad as the **worst single domain**: any High -> overall High;
else any Some concerns -> Some concerns; else Low. One fatal flaw can invalidate the result no
matter how clean everything else is — the loaded-coin logic.

In [4]:
def overall_rob2(domains: dict) -> Judg:
    """WORST-DOMAIN logic: overall is as bad as the worst single domain."""
    return Judg(max(domains.values()))     # HIGH(2) beats SOME(1) beats LOW(0)


def assess_rob2(answers: dict) -> dict:
    d = {
        "D1_randomization": d1_randomization(answers),
        "D2_deviations":    d2_deviations(answers),
        "D3_missing_data":  d3_missing_data(answers),
        "D4_measurement":   d4_measurement(answers),
        "D5_reported":      d5_reported_result(answers),
    }
    overall = overall_rob2(d)
    return {
        "domains": {k: LABEL[v] for k, v in d.items()},
        "overall": LABEL[overall],
        "driven_by": [k for k, v in d.items() if v == overall and overall == Judg.HIGH],
    }

print("assess_rob2() ready — answers in, per-domain + overall out")

assess_rob2() ready — answers in, per-domain + overall out


## Worked example — an open-label physio trial, per outcome

Physiotherapy vs usual care. Randomization is fine and dropout is low, but patients know their
arm (open-label). For the **self-reported pain** outcome, D4 fires **High** (subjective outcome,
unblinded patient) and worst-domain logic makes the whole trial **High**.

In [5]:
# Open-label physio trial, SELF-REPORTED PAIN outcome
pain = assess_rob2({
    "random_sequence": "Y", "allocation_concealed": "Y", "baseline_balanced": "Y",
    "participants_aware": "Y", "deviations_affected_outcome": "PN",
    "data_for_nearly_all": "Y", "missingness_related_to_outcome": "N",
    "assessor_blinded": "N", "outcome_subjective": "Y",       # <- the killer
    "prespecified": "Y", "selected_from_many": "N",
})
for k, v in pain["domains"].items():
    print(f"  {k:<18}: {v}")
print("PAIN   :", pain["overall"], "driven by", pain["driven_by"])  # High / D4

  D1_randomization  : Low
  D2_deviations     : Some concerns
  D3_missing_data   : Low
  D4_measurement    : High
  D5_reported       : Low
PAIN   : High driven by ['D4_measurement']


## Same trial, different outcome — one answer flips it

Now score the **mortality** outcome. Death is objective — hard to mis-measure regardless of
blinding — so the *only* change is `outcome_subjective: "Y" -> "N"`. D4 drops to Low, and the
worst domain becomes D2's **Some concerns**. Same trial, different risk of bias per outcome —
exactly why RoB2 is applied **per outcome**.

In [6]:
# SAME trial, MORTALITY outcome (objective) -> D4 no longer High
death = assess_rob2({
    "random_sequence": "Y", "allocation_concealed": "Y", "baseline_balanced": "Y",
    "participants_aware": "Y", "deviations_affected_outcome": "PN",
    "data_for_nearly_all": "Y", "missingness_related_to_outcome": "N",
    "assessor_blinded": "N", "outcome_subjective": "N",       # objective outcome
    "prespecified": "Y", "selected_from_many": "N",
})
for k, v in death["domains"].items():
    print(f"  {k:<18}: {v}")
print("DEATH  :", death["overall"])                            # Some concerns (D2)
print("\nOnly ONE answer changed (outcome_subjective) -> the rule re-derived the verdict.")

  D1_randomization  : Low
  D2_deviations     : Some concerns
  D3_missing_data   : Low
  D4_measurement    : Some concerns
  D5_reported       : Low
DEATH  : Some concerns

Only ONE answer changed (outcome_subjective) -> the rule re-derived the verdict.


## Six trials, one outcome — worst domain wins every time

A review pools six RCTs for "severe hypoglycemia." Each trial is scored per this outcome. Half
come out **High** — each for a *different* reason (differential dropout, no concealment, a
post-hoc outcome switch). One High domain sinks the whole trial regardless of its clean domains,
which is what triggers a GRADE risk-of-bias downgrade downstream.

In [7]:
clean = {
    "random_sequence": "Y", "allocation_concealed": "Y", "baseline_balanced": "Y",
    "participants_aware": "N", "deviations_affected_outcome": "N",
    "data_for_nearly_all": "Y", "missingness_related_to_outcome": "N",
    "assessor_blinded": "Y", "outcome_subjective": "N",
    "prespecified": "Y", "selected_from_many": "N",
}

def variant(**over):
    a = dict(clean); a.update(over); return a

trials = {
    "A": variant(),                                                       # clean -> Low
    "B": variant(participants_aware="Y"),                                # unblinded, no dev -> Some
    "C": variant(data_for_nearly_all="N", missingness_related_to_outcome="Y"),  # dropout -> High (D3)
    "D": variant(),                                                       # clean -> Low
    "E": variant(allocation_concealed="N"),                              # no concealment -> High (D1)
    "F": variant(participants_aware="Y", prespecified="N", selected_from_many="Y"),  # post-hoc -> High (D5)
}

print(f"{'Trial':<6}{'D1':<6}{'D2':<6}{'D3':<6}{'D4':<6}{'D5':<6}{'Overall'}")
print("-" * 46)
n_high = 0
for name, ans in trials.items():
    r = assess_rob2(ans)
    dom = r["domains"]
    short = {"Low": "Low", "Some concerns": "Some", "High": "High"}
    row = [short[dom[k]] for k in
           ("D1_randomization","D2_deviations","D3_missing_data","D4_measurement","D5_reported")]
    if r["overall"] == "High":
        n_high += 1
    print(f"{name:<6}" + "".join(f"{x:<6}" for x in row) + r["overall"])

print(f"\n{n_high} of {len(trials)} trials are High risk, each for a DIFFERENT reason.")
print("-> in GRADE this triggers a risk-of-bias downgrade, pulling certainty toward LOW.")

Trial D1    D2    D3    D4    D5    Overall
----------------------------------------------
A     Low   Low   Low   Low   Low   Low
B     Low   Some  Low   Low   Low   Some concerns
C     Low   Low   High  Low   Low   High
D     Low   Low   Low   Low   Low   Low
E     High  Low   Low   Low   Low   High
F     Low   Some  Low   Low   High  High

3 of 6 trials are High risk, each for a DIFFERENT reason.
-> in GRADE this triggers a risk-of-bias downgrade, pulling certainty toward LOW.


## Recap

- **Risk of bias = systematic error** from how a study was run — a loaded coin, not random noise.
- The universal pattern: **signaling questions -> fixed algorithm -> domain judgment -> overall**; RoB2 & ROBINS-I use **worst-domain** logic (`max()` over an ordered enum).
- You watched one signaling answer (`outcome_subjective`) flip a trial from **High** (pain) to **Some concerns** (mortality) — RoB2 is **per outcome**.
- One High domain sinks a whole trial; the reviewer answers the questions, the code applies the rule identically every time.

← Back to the **Risk of Bias** article for ROBINS-I, AMSTAR-2, the full domain tables, and interview Q&A.